# Advanced RAG with Local TinyLlama + Ragas

**Environment:** Python 3.12  
**LangChain:** 1.3.14  
**Ragas:** 0.4.3  
**Generator / evaluator:** local `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

### What this notebook does

This notebook keeps the useful parts of the previous implementation and removes the experimental Gemini/Ollama blocks.

Pipeline:

**PDF → Markdown → clean text → real headings → structure-aware semantic chunks → ChromaDB → dense retrieval + BM25 → hybrid retrieval → cross-encoder reranking → TinyLlama generation → Ragas evaluation**

### Important Ragas point

Ragas does **not** require OpenAI specifically. It needs an **evaluator LLM** for metrics such as Faithfulness, Answer Relevancy, Context Precision and Context Recall.

Here the evaluator is also local TinyLlama, so no `OPENAI_API_KEY`, Gemini key, or other cloud API key is used.

Because TinyLlama is a small 1.1B model, its Ragas scores should be treated as a learning/demo evaluation rather than a highly reliable benchmark.

## 1. Configuration

Keep paths and retrieval parameters in one place so the rest of the notebook is easy to follow.

In [ ]:
from pathlib import Path
import json
import re
import numpy as np

RAW_PDF = Path("data/raw/MachineLearningTomMitchell.pdf")
MD_PATH = Path("data/processed/ml_book.md")
CLEAN_PATH = Path("data/processed/clean_ml_book.md")
CHUNKS_PATH = Path("data/processed/chunks.jsonl")
CHROMA_PATH = "data/chroma"

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LOCAL_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

TOP_K_HYBRID = 10
TOP_K_FINAL = 5

MD_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")

## 2. Check the environment

This is only a version check. It does not reinstall packages.

The notebook is written for the versions you provided: Python 3.12, LangChain 1.3.14 and Ragas 0.4.3.

In [ ]:
import sys
import langchain
import ragas

print("Python:", sys.version)
print("LangChain:", langchain.__version__)
print("Ragas:", ragas.__version__)

assert sys.version_info[:2] == (3, 12), "This notebook was prepared for Python 3.12."
assert langchain.__version__ == "1.3.14", "Expected LangChain 1.3.14."
assert ragas.__version__ == "0.4.3", "Expected Ragas 0.4.3."

print("Version check passed.")

## 3. PDF → Markdown

The extraction is cached. If the Markdown file already exists, the expensive PDF extraction is skipped.

In [ ]:
if MD_PATH.exists():
    print(f"Already extracted: {MD_PATH}")
else:
    import pymupdf
    import pymupdf4llm
    from tqdm import tqdm

    doc = pymupdf.open(RAW_PDF)
    print(f"Total pages: {len(doc)}")

    all_pages = []
    for page_num in tqdm(range(len(doc)), desc="Extracting PDF", unit="page"):
        page_data = pymupdf4llm.to_markdown(
            doc,
            pages=[page_num],
            page_chunks=True
        )
        all_pages.extend(page_data)

    doc.close()

    with open(MD_PATH, "w", encoding="utf-8") as f:
        for page_number, page in enumerate(all_pages, start=1):
            f.write(f"\n\n<!-- PAGE {page_number} -->\n\n")
            f.write(page["text"])

    print(f"Saved to: {MD_PATH}")

In [ ]:
text = MD_PATH.read_text(encoding="utf-8")

pages = re.findall(r"<!-- PAGE (\d+) -->", text)

print(f"Characters: {len(text):,}")
print(f"Words: {len(text.split()):,}")
print(f"Page markers: {len(pages)}")

## 4. Clean the extracted Markdown

We keep the page markers because page numbers are useful metadata later when showing the source of an answer.

In [ ]:
if CLEAN_PATH.exists():
    print(f"Already cleaned: {CLEAN_PATH}")
else:
    text = MD_PATH.read_text(encoding="utf-8")

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\n*\s*(<!-- PAGE \d+ -->)\s*\n*", r"\n\n\1\n\n", text)
    text = re.sub(r" +([,.!?;:])", r"\1", text)

    CLEAN_PATH.write_text(text.strip(), encoding="utf-8")
    print(f"Saved to: {CLEAN_PATH}")

text = CLEAN_PATH.read_text(encoding="utf-8")
print(f"Cleaned characters: {len(text):,}")

## 5. Keep only real section headings

`pymupdf4llm` can turn bold text, captions and other emphasized text into Markdown headings.

We therefore keep:
- numbered section headings such as `3.7.1 ...`
- genuine standalone ALL-CAPS headings

Figure and table captions are explicitly rejected.

In [ ]:
HEADING_RE = re.compile(r"(?m)^(#{1,6})\s+(.+?)\s*$")
PAGE_RE = re.compile(r"<!-- PAGE (\d+) -->")

def strip_md_emphasis(s: str) -> str:
    return re.sub(r"[*_]+", "", s).strip()

def is_real_heading(raw_title: str) -> bool:
    title = strip_md_emphasis(raw_title)

    if re.match(r"^(FIGURE|TABLE)\b", title, re.IGNORECASE):
        return False

    if re.match(r"^\d+(\.\d+){0,4}\s+\S", title):
        return True

    letters_only = re.sub(r"[^A-Za-z]", "", title)
    if len(letters_only) >= 4 and letters_only.isupper():
        return True

    return False

structure = []
current_page = None

for line in text.splitlines():
    page_match = PAGE_RE.match(line.strip())
    if page_match:
        current_page = int(page_match.group(1))
        continue

    heading_match = re.match(r"^(#{1,6})\s+(.+?)\s*$", line.strip())
    if heading_match:
        level = len(heading_match.group(1))
        raw_title = heading_match.group(2)

        if is_real_heading(raw_title):
            structure.append({
                "page": current_page,
                "level": level,
                "title": strip_md_emphasis(raw_title),
            })

print(f"Real headings kept: {len(structure)}")

for item in structure[:20]:
    print(f"p{item['page']:>3} | {'  ' * (item['level'] - 1)}{item['title']}")

## 6. Split the book into structure-aware sections

Each section keeps its heading path and starting page. This metadata will later be attached to every retrieved document.

In [ ]:
def find_heading_positions(text, structure):
    positions = []
    search_from = 0

    for item in structure:
        pattern = re.compile(
            r"(?m)^#{1,6}\s+\*{0,3}_{0,3}" +
            re.escape(item["title"][:40])
        )
        match = pattern.search(text, search_from)

        if match is None:
            match = pattern.search(text)

        positions.append(match.start() if match else search_from)

        if match:
            search_from = match.end()

    return positions

def heading_path(idx):
    level = structure[idx]["level"]
    path = [structure[idx]["title"]]

    for j in range(idx - 1, -1, -1):
        if structure[j]["level"] < level:
            path.insert(0, structure[j]["title"])
            level = structure[j]["level"]

        if level <= 1:
            break

    return " > ".join(path)

positions = find_heading_positions(text, structure)

sections = []

for i, item in enumerate(structure):
    start = positions[i]
    end = positions[i + 1] if i + 1 < len(positions) else len(text)

    body = text[start:end]
    body = re.sub(r"(?m)^#{1,6}\s+.+?$", "", body, count=1).strip()
    body = PAGE_RE.sub("", body).strip()

    if body:
        sections.append({
            "title": item["title"],
            "heading_path": heading_path(i),
            "page": item["page"],
            "text": body,
        })

print(f"Sections with body text: {len(sections)}")

## 7. Semantic chunking

We use LangChain's `SemanticChunker` inside each real section.

A fallback recursive splitter is used before semantic chunking for very large sections. This prevents one enormous section from becoming an unnecessarily expensive embedding operation.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

import torch

embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", embedding_device)

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": embedding_device},
    encode_kwargs={"normalize_embeddings": True},
)

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)

fallback_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

MIN_CHUNK_CHARS = 200
MAX_SEMANTIC_INPUT = 20000

In [ ]:
merged_sections = []
carry = ""

for sec in sections:
    combined = (carry + "\n\n" + sec["text"]).strip() if carry else sec["text"]

    if len(combined) < MIN_CHUNK_CHARS:
        carry = combined
        continue

    merged_sections.append({**sec, "text": combined})
    carry = ""

if carry:
    if merged_sections:
        merged_sections[-1]["text"] += "\n\n" + carry
    else:
        merged_sections.append({**sections[-1], "text": carry})

print(
    f"Sections before merge: {len(sections)} -> "
    f"after merge: {len(merged_sections)}"
)

In [ ]:
def chunk_section_text(section_text):
    pieces = (
        fallback_splitter.split_text(section_text)
        if len(section_text) > MAX_SEMANTIC_INPUT
        else [section_text]
    )

    chunks_out = []

    for piece in pieces:
        chunks_out.extend(semantic_splitter.split_text(piece))

    return chunks_out

chunks = []

for sec in merged_sections:
    for piece in chunk_section_text(sec["text"]):
        piece = piece.strip()

        if not piece:
            continue

        chunks.append({
            "chunk_id": len(chunks),
            "heading_path": sec["heading_path"],
            "page": sec["page"],
            "text": piece,
            "n_chars": len(piece),
        })

print(f"Total chunks: {len(chunks):,}")

lengths = [c["n_chars"] for c in chunks]
print(f"Min: {min(lengths)}")
print(f"Max: {max(lengths)}")
print(f"Average: {sum(lengths) / len(lengths):.0f}")

## 8. Save the chunks

Saving the chunks lets us rebuild the vector database without repeating PDF extraction and semantic chunking.

In [ ]:
with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Saved {len(chunks):,} chunks to {CHUNKS_PATH}")

## 9. Create LangChain Documents

ChromaDB stores the chunk text while the metadata keeps the section and page information.

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=chunk["text"],
        metadata={
            "chunk_id": chunk["chunk_id"],
            "heading_path": chunk["heading_path"],
            "page": chunk["page"],
        },
    )
    for chunk in chunks
]

print(f"Documents ready: {len(documents):,}")

## 10. Build the ChromaDB vector store

Dense retrieval finds chunks that are semantically similar to the query.

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="ml_book",
    persist_directory=CHROMA_PATH,
)

print("ChromaDB created successfully.")

## 11. Dense retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_HYBRID},
)

query = "What is machine learning?"
dense_docs = retriever.invoke(query)

for i, doc in enumerate(dense_docs, 1):
    print(f"\n--- Dense result {i} ---")
    print(doc.page_content[:350])
    print(doc.metadata)

## 12. BM25 lexical retrieval

Dense retrieval is good for meaning, while BM25 is good at exact terms.

We combine both later with Reciprocal Rank Fusion (RRF).

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_docs = [
    doc.page_content.lower().split()
    for doc in documents
]

bm25 = BM25Okapi(tokenized_docs)

print("BM25 index created.")

## 13. Hybrid search with Reciprocal Rank Fusion

A document receives a score from both dense retrieval and BM25. RRF combines their rankings without requiring the two raw scores to be comparable.

In [ ]:
def hybrid_search(query, k=TOP_K_HYBRID, fetch_k=20):
    dense_docs = retriever.invoke(query)[:fetch_k]

    query_tokens = query.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_indices = bm25_scores.argsort()[-fetch_k:][::-1]

    rrf_scores = {}

    for rank, doc in enumerate(dense_docs):
        doc_id = doc.metadata["chunk_id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (60 + rank + 1)

    for rank, idx in enumerate(bm25_indices):
        doc_id = documents[idx].metadata["chunk_id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1 / (60 + rank + 1)

    ranked_ids = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:k]

    doc_lookup = {
        doc.metadata["chunk_id"]: doc
        for doc in documents
    }

    return [doc_lookup[doc_id] for doc_id in ranked_ids]

In [ ]:
results = hybrid_search("What is machine learning?", k=5)

for i, doc in enumerate(results, 1):
    print(f"\n--- Hybrid result {i} ---")
    print(doc.page_content[:350])

## 14. Cross-encoder reranking

Hybrid search gives us a candidate set. The cross-encoder reads the query and each candidate together and produces a more direct relevance score.

We retrieve more candidates first, then keep the best few.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANKER_MODEL)

def rerank(query, docs, top_k=TOP_K_FINAL):
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [doc for _, doc in ranked[:top_k]]

In [ ]:
query = "What is machine learning?"

hybrid_docs = hybrid_search(query, k=TOP_K_HYBRID)
final_docs = rerank(query, hybrid_docs, top_k=TOP_K_FINAL)

for i, doc in enumerate(final_docs, 1):
    print(f"\n--- Final context {i} ---")
    print(doc.page_content[:400])
    print(doc.metadata)

## 15. Build the RAG prompt

The generator receives only the reranked context.

The source tag is included in the context so TinyLlama can cite the section/page when it answers.

In [ ]:
def build_context(docs):
    parts = []

    for doc in docs:
        meta = doc.metadata
        source = (
            f"[{meta.get('heading_path', 'Unknown section')}, "
            f"p.{meta.get('page', '?')}]"
        )
        parts.append(f"{source}\n{doc.page_content}")

    return "\n\n".join(parts)

def build_prompt(context, question):
    return f"""<|system|>
You are a question-answering assistant.
Answer using ONLY the supplied context.
If the answer is not present in the context, say:
"I don't know based on the provided context."
Keep the answer concise.
Cite the relevant source tag after important claims.
<|user|>
Context:
{context}

Question:
{question}
<|assistant|>
"""

## 16. Load TinyLlama once

The earlier notebook created the local model inside `rag()`.

That is inefficient because every question reloads the model.

Here we load TinyLlama exactly once and reuse the same model for:
1. RAG generation
2. Ragas evaluation

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

llm_device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if llm_device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL,
    torch_dtype=dtype,
    device_map="auto" if llm_device == "cuda" else None,
)

if llm_device == "cpu":
    model = model.to("cpu")

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)

hf_llm = HuggingFacePipeline(pipeline=text_pipeline)

# ChatHuggingFace lets LangChain/Ragas treat TinyLlama as a chat model.
local_llm = ChatHuggingFace(llm=hf_llm)

print("TinyLlama loaded.")

## 17. Complete RAG function

The complete flow is now:

**query → hybrid retrieval → reranking → context → TinyLlama**

In [ ]:
def extract_response_text(response):
    content = response.content

    if isinstance(content, str):
        return content.strip()

    if isinstance(content, list):
        texts = []

        for block in content:
            if isinstance(block, dict) and "text" in block:
                texts.append(block["text"])

        return "".join(texts).strip()

    return str(content).strip()


def rag(query):
    hybrid_docs = hybrid_search(
        query,
        k=TOP_K_HYBRID,
        fetch_k=20,
    )

    final_docs = rerank(
        query,
        hybrid_docs,
        top_k=TOP_K_FINAL,
    )

    context = build_context(final_docs)
    prompt_text = build_prompt(context, query)

    response = local_llm.invoke(prompt_text)
    answer = extract_response_text(response)

    return answer, final_docs

In [ ]:
question = "What is machine learning?"

answer, sources = rag(question)

print("QUESTION:")
print(question)

print("\nANSWER:")
print(answer)

print("\nSOURCES:")
for i, doc in enumerate(sources, 1):
    print(f"{i}. {doc.metadata}")

## 18. Prepare a small evaluation set

A Ragas evaluation dataset needs the question, generated response and retrieved contexts.

For Context Recall and other reference-based metrics, we also provide a reference answer.

In [ ]:
eval_set = [
    {
        "question": "What is machine learning?",
        "reference": (
            "Machine learning is the study of computer programs that "
            "improve their performance on a task through experience."
        ),
    },
    {
        "question": "What is supervised learning?",
        "reference": (
            "Supervised learning is learning a function that maps inputs "
            "to outputs from a set of labeled training examples."
        ),
    },
    {
        "question": "What is unsupervised learning?",
        "reference": (
            "Unsupervised learning is learning patterns or structure "
            "from data that has no labeled outputs."
        ),
    },
    {
        "question": "What is reinforcement learning?",
        "reference": (
            "Reinforcement learning is learning what actions to take, "
            "given a state, in order to maximize a numerical reward signal over time."
        ),
    },
    {
        "question": "What are the main applications of machine learning?",
        "reference": (
            "Machine learning is applied in areas such as data mining, "
            "speech and image recognition, fraud detection, autonomous "
            "vehicles, and information-filtering systems."
        ),
    },
]

In [ ]:
eval_rows = []

for item in eval_set:
    answer, sources = rag(item["question"])

    eval_rows.append({
        "user_input": item["question"],
        "response": answer,
        "retrieved_contexts": [doc.page_content for doc in sources],
        "reference": item["reference"],
    })

print(f"Evaluation samples: {len(eval_rows)}")

## 19. Connect Ragas to the local TinyLlama

This is the important part.

The old code created:

```python
AsyncOpenAI()
llm_factory("gpt-4o-mini", ...)
```

That explicitly creates an OpenAI evaluator.

We do **not** do that here.

Instead, Ragas receives the LangChain TinyLlama model through `LangchainLLMWrapper`.

Ragas 0.4.3 has moved toward its newer native LLM interfaces and marks `LangchainLLMWrapper` as deprecated, but the wrapper is still supported in 0.4.x. It is useful here because our model is already a LangChain local model and we want to keep the setup simple.

In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm = LangchainLLMWrapper(local_llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)

print("Local TinyLlama connected to Ragas.")

## 20. Ragas 0.4.3 metrics

Ragas 0.4 moved the core metrics to `ragas.metrics.collections`.

- **Faithfulness:** Is the answer supported by the retrieved context?
- **Answer Relevancy:** Does the answer address the question?
- **Context Precision:** Are the retrieved contexts relevant/ranked well?
- **Context Recall:** Did retrieval contain information needed by the reference answer?

`AnswerRelevancy` also needs an embeddings model, so we give Ragas the same local BGE embedding model used by our retriever.

In [ ]:
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
)

faithfulness_metric = Faithfulness(llm=evaluator_llm)

answer_relevancy_metric = AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

context_precision_metric = ContextPrecision(llm=evaluator_llm)

context_recall_metric = ContextRecall(llm=evaluator_llm)

metrics = {
    "faithfulness": faithfulness_metric,
    "answer_relevancy": answer_relevancy_metric,
    "context_precision": context_precision_metric,
    "context_recall": context_recall_metric,
}

print("Ragas metrics ready.")

## 21. Run the Ragas evaluation

Ragas 0.4 metrics expose the `ascore()` interface.

Using the metric objects directly also makes it clear which inputs each metric receives and avoids mixing the older v0.3 API with the v0.4 collections API.

In [ ]:
import pandas as pd

async def score_faithfulness(row):
    result = await faithfulness_metric.ascore(
        user_input=row["user_input"],
        response=row["response"],
        retrieved_contexts=row["retrieved_contexts"],
    )
    return float(result.value)

async def score_answer_relevancy(row):
    result = await answer_relevancy_metric.ascore(
        user_input=row["user_input"],
        response=row["response"],
    )
    return float(result.value)

async def score_context_precision(row):
    result = await context_precision_metric.ascore(
        user_input=row["user_input"],
        retrieved_contexts=row["retrieved_contexts"],
        reference=row["reference"],
    )
    return float(result.value)

async def score_context_recall(row):
    result = await context_recall_metric.ascore(
        user_input=row["user_input"],
        retrieved_contexts=row["retrieved_contexts"],
        reference=row["reference"],
    )
    return float(result.value)

results = []

for row in eval_rows:
    print(f"Evaluating: {row['user_input']}")

    row_result = {
        "question": row["user_input"],
        "faithfulness": await score_faithfulness(row),
        "answer_relevancy": await score_answer_relevancy(row),
        "context_precision": await score_context_precision(row),
        "context_recall": await score_context_recall(row),
    }

    results.append(row_result)

results_df = pd.DataFrame(results)
results_df

## 22. Average Ragas scores

Scores are between 0 and 1, where higher is generally better.

Do not treat TinyLlama's evaluation scores as ground truth. The evaluator is itself a small local model, so the scores are mainly useful for learning the evaluation workflow and comparing changes within this project.

In [ ]:
score_columns = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]

results_df[score_columns].mean().sort_values(ascending=False)

## 23. What each part of the project is doing

| Component | Purpose |
|---|---|
| BGE embeddings | Converts chunks and queries into vectors |
| ChromaDB | Stores vectors and performs dense retrieval |
| BM25 | Finds exact lexical matches |
| RRF | Combines dense + BM25 rankings |
| Cross-encoder | Reranks the strongest candidates |
| TinyLlama | Generates the final answer |
| Ragas + TinyLlama | Evaluates the generated answer and retrieved context |

### One important distinction

The **embedding model is not the generation model**.

BGE is used for retrieval and Answer Relevancy's embedding-based part.

TinyLlama is used for generation and as the Ragas evaluator.

No OpenAI API is involved in this notebook.

## 24. Final pipeline

```text
                         ┌── BGE Embeddings ──→ ChromaDB ──→ Dense Search ──┐
PDF → Cleaning → Chunks ─┤                                                   ├→ RRF
                         └── BM25 ───────────────→ Keyword Search ───────────┘
                                                        ↓
                                                  Cross Encoder
                                                        ↓
                                               Top 5 Contexts
                                                        ↓
                                                   TinyLlama
                                                        ↓
                                                     Answer
                                                        ↓
                                           Ragas + Local TinyLlama
                                                        ↓
                         Faithfulness / Answer Relevancy / Context Precision / Context Recall
```

### What was removed from the previous notebook

- Gemini generation code
- Ollama alternative code
- OpenAI Ragas evaluation code
- duplicated `build_context()` definition
- duplicated/obsolete Ragas dataset blocks
- unused optional NetworkX graph block
- commented-out experimental cells

The core Advanced RAG pieces—semantic chunking, ChromaDB, BM25 hybrid search and reranking—are retained.

## 25. If Ragas fails with a parsing/structured-output error

The most likely limitation is **TinyLlama**, not an API key.

Ragas asks its evaluator LLM to produce structured evaluation information. TinyLlama 1.1B is a very small model and may occasionally fail to follow that evaluation format.

If this happens, first check the generation output itself. The RAG pipeline can still work even when the Ragas judge struggles.

For a stronger evaluation later, keep the RAG generator local but use a stronger local evaluator model. That changes only the evaluator model; the retrieval pipeline does not need to change.